## HW5: UFO Sighthings
# Fernando Degante

This notebook uses the UFO sightings dataset to create two visualizations: reported sightings by year and reported shapes by country. The second chart includes a country dropdown.

In [2]:
import pandas as pd
import altair as alt

data_url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

columns = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_text", "comments",
    "date_posted", "latitude", "longitude"
]

df = pd.read_csv(
    data_url,
    header=None,
    names=columns,
    low_memory=False
)

print("Rows and columns:", df.shape)
df.head()

Rows and columns: (80332, 11)


,datetime,city,state,country,shape,duration_seconds,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [5]:
df["year"] = df["datetime"].str.split().str[0].str.split("/").str[2]
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df[["datetime", "year", "shape"]].head()

,datetime,year,shape
0,10/10/1949 20:30,1949,cylinder
1,10/10/1949 21:00,1949,light
2,10/10/1955 17:00,1955,circle
3,10/10/1956 21:00,1956,circle
4,10/10/1960 20:00,1960,light


In [7]:
year_counts = df.groupby("year").size().reset_index(name="sightings")

chart1 = alt.Chart(year_counts).mark_line().encode(
    x=alt.X("year:Q", axis=alt.Axis(format="d")),
    y="sightings:Q"
).properties(width=600, title="Reported UFO Sightings by Year")

chart1

alt.Chart(...)

In [8]:
shape_counts = df.groupby(["country", "shape"]).size().reset_index(name="sightings")

shape_counts.head()

,country,shape,sightings
0,au,changing,9
1,au,chevron,3
2,au,cigar,15
3,au,circle,62
4,au,cone,6


In [9]:
dropdown = alt.binding_select(
    options=sorted(shape_counts["country"].unique()),
    name="Country: "
)

country = alt.param(value="us", bind=dropdown)

chart2 = alt.Chart(shape_counts).mark_bar().encode(
    x="sightings:Q",
    y=alt.Y("shape:N", sort="-x")
).add_params(
    country
).transform_filter(
    alt.datum.country == country
).properties(title="Reported UFO Shapes by Country")

chart2

alt.Chart(...)

In [10]:
chart1.save("chart1.json")
chart2.save("chart2.json")